In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split
import sympy as sp
import random
from torch.utils.data import DataLoader, Subset

config

In [ ]:
Initial_training = 20
train_pool = 20
BATCH_SIZE = 5
LEARNING_RATE = 1e-3
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs_active = 3

data loading

In [ ]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# Data transforms
# ----------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010)
    )
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010)
    )
])

# ----------------------------
# Base datasets
# ----------------------------
# Use augmentation for labeled training data
train_dataset_aug = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

# Use deterministic transforms for pool evaluation
train_dataset_eval = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=eval_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=eval_transform
)

# ----------------------------
# Split indices manually
# ----------------------------
all_indices = list(range(len(train_dataset_aug)))
random.shuffle(all_indices)

train_indices = all_indices[:Initial_training]
pool_indices = all_indices[Initial_training:(Initial_training + train_pool)]
rest_indices = all_indices[Initial_training + train_pool:]  # optional, unused here

# ----------------------------
# Build subsets/loaders
# ----------------------------
def build_loaders(train_indices, pool_indices, batch_size):
    labeled_train_set = Subset(train_dataset_aug, train_indices)
    pool_set = Subset(train_dataset_eval, pool_indices)

    train_loader = DataLoader(
        labeled_train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    # IMPORTANT: shuffle=False so pool index positions stay stable for QBC
    pool_loader = DataLoader(
        pool_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return labeled_train_set, pool_set, train_loader, pool_loader, test_loader

Initial_train, pool, train_loader, pool_loader, test_loader = build_loaders(
    train_indices, pool_indices, BATCH_SIZE
)

# ----------------------------
# Move one selected point from pool -> labeled set
# ----------------------------
def add_pool_point_to_training(selected_pool_idx, train_indices, pool_indices):
    """
    selected_pool_idx:
        Index relative to the current pool subset ordering
        (e.g. argmax over QBC scores computed with pool_loader, shuffle=False)

    Returns updated train_indices, pool_indices
    """
    original_idx = pool_indices[selected_pool_idx]

    train_indices.append(original_idx)
    del pool_indices[selected_pool_idx]

    return train_indices, pool_indices

# ----------------------------
# Example usage after QBC
# ----------------------------
# Suppose QBC selected the sample at position 17 in the pool subset:
# selected_pool_idx = 17
#
# train_indices, pool_indices = add_pool_point_to_training(
#     selected_pool_idx,
#     train_indices,
#     pool_indices
# )
#
# Initial_train, pool, train_loader, pool_loader, test_loader = build_loaders(
#     train_indices, pool_indices, BATCH_SIZE
# )

# ----------------------------
# Class names
# ----------------------------
classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),   # 32x32 -> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),   # 16x16 -> 8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)    # 8x8 -> 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


Train

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

evaluate

In [ ]:
def evaluate(model, loader, criterion, device):

    with torch.no_grad():
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total
        return epoch_loss, epoch_acc

In [ ]:
# ----------------------------
# Save model
# ----------------------------
torch.save(model.state_dict(), "cifar10_cnn.pth")
print("Model saved to cifar10_cnn.pth")

In [ ]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
models = [model]

In [ ]:
def train(models,epochs,train_loader,criterion,optimizer):
    for model in models:
        for epoch in range(epochs):
            _, _ = train_one_epoch(
                model, train_loader, criterion, optimizer, DEVICE
            )

In [ ]:
#pre train
train(models,EPOCHS,train_loader,criterion,optimizer)


#active loop

for i in range(epochs_active):
    model_pred = [[] for i in range(len(models))]
    for i, model in enumerate(models):
        model.to(DEVICE)
        model.eval()
        with torch.no_grad():
            for image, label in pool:
                image = image.unsqueeze(0).to(DEVICE)  # (1, 3, 32, 32)
                out = model(image)                     # (1, 10) logits
                probs = torch.softmax(out, dim=1)      # softmax over class dimension

                pred_idx = probs.argmax(dim=1).item()
                pred_label = classes[pred_idx]
                model_pred[i].append(pred_label)
    QBC_predictions = []
    QBC_vote_entropy = []
    for i in range(len(pool)):
        counts = []
        vote_entropy = []
        for C in classes:
            count = 0
            for model in model_pred:
                if model[i] == C:
                    count += 1
            counts.append((count,C))

            p = count / len(models)
            term = p * sp.log(p, 2) if p > 0 else 0
            vote_entropy.append(term)
        
        QBC_predictions.append(max(counts))
        QBC_vote_entropy.append((sum(vote_entropy),i))
    

    selected_pool_idx = max(QBC_vote_entropy)[1]
    
    train_indices, pool_indices = add_pool_point_to_training(
        selected_pool_idx,
        train_indices,
        pool_indices
    )
    
    Initial_train, pool, train_loader, pool_loader, test_loader = build_loaders(
        train_indices, pool_indices, BATCH_SIZE
    )
    train(models,1,train_loader,criterion,optimizer)

